# AQD operational pipeline

Use this notebook to run the AQD workflow end-to-end: `proc_1` reads the deployment, writes review plots, and trims to the deployment window; `proc_2` applies any manual QC flags to the trimmed `proc_1` file; `imos_delivery` publishes FV00 from `proc_1` and FV01 from `proc_2`.

**Operator choices**
- Run the full pipeline by leaving all step toggles enabled.
- Run `proc_1` only to regenerate the trimmed file and review plots.
- Rerun `proc_2` with updated `manual_qc_flags` to refresh QC outputs without repeating `proc_1`.
- Rerun delivery only when `proc_1` and `proc_2` outputs already exist.


## Imports

This cell prepares the notebook path and imports the workflow entry points used in the later steps.


In [ ]:
import sys
from pathlib import Path

try:
    from IPython.display import Image, display
except ModuleNotFoundError:
    Image = None

    def display(value):
        print(value)

NOTEBOOK_DIR = Path.cwd().resolve()
if not (NOTEBOOK_DIR / 'tools').exists() and (NOTEBOOK_DIR / 'mooring_proc' / 'tools').exists():
    sys.path.insert(0, str((NOTEBOOK_DIR / 'mooring_proc').resolve()))

from tools.database_lookup import list_instruments
from tools.workflows.run_imos_delivery import run_imos_delivery
from tools.workflows.run_proc1 import run_proc1
from tools.workflows.run_proc2 import run_proc2


## User options

Set the deployment identifier, optional metadata preview filters, and step toggles in this single configuration cell before running any workflow step.


In [ ]:
# Optional metadata CSV path. Set this to an absolute path before running preview or workflow steps.
metadata_csv = None

# Required AQD deployment identifier from the metadata table.
inst_deploy_id = ""

# Keep this notebook scoped to AQD instruments.
instrument = "AQD"

# Optional metadata preview filters.
show_year = None
show_location = None

# Preview and execution toggles.
show_table_preview = True
run_proc1_step = True
run_proc2_step = True
run_delivery_step = True
show_plots = False

# Template manual QC window for proc_2. Duplicate or edit entries as needed.
manual_qc_flags = [
    {
        "qc_vars": ["UCUR_quality_control", "VCUR_quality_control"],
        "flag": 3,
        "start": "2024-07-31 06:00:00",
        "end": "2024-08-01 00:00:00",
        "comment": "Example manual QC window - replace with deployment-specific values.",
    }
]


This cell turns the options above into the shared workflow configuration and initializes output tracking for the rest of the notebook.


In [ ]:
def build_workflow_config():
    return {
        "metadata_csv": metadata_csv,
        "inst_deploy_ID": inst_deploy_id,
        "instrument": instrument,
        "manual_qc_flags": manual_qc_flags,
    }


def require_ready_config(require_metadata=True, require_instrument=True):
    if require_metadata and not metadata_csv:
        raise ValueError("Set metadata_csv to an absolute metadata CSV path before running this cell.")
    if require_instrument and not str(inst_deploy_id).strip():
        raise ValueError("Set inst_deploy_id to a valid AQD deployment identifier before running this cell.")
    return build_workflow_config()


proc1_result = None
proc2_result = None
delivery_result = None
summary = {"proc1": None, "proc2": None, "qc_log": None, "fv00": None, "fv01": None}


## Optional metadata preview

If enabled, this cell lists AQD deployments from the metadata table so you can confirm the correct `inst_deploy_id` before processing.


In [ ]:
preview_table = None
if show_table_preview:
    require_ready_config(require_metadata=True, require_instrument=False)
    preview_table = list_instruments(
        metadata_csv,
        year=show_year,
        location=show_location,
        instrument=instrument,
    )
    if preview_table.empty:
        print("No AQD deployments matched the preview filters. Clear show_year/show_location or check metadata_csv.")
    else:
        display(preview_table)
else:
    print("Metadata preview skipped.")


if run_proc1_step:
    workflow_config = require_ready_config()
    # Expected outputs: a trimmed proc_1 NetCDF plus pre/post deployment review plots.
    proc1_result = run_proc1(workflow_config)
    summary['proc1'] = proc1_result['output_path']
    print(f"proc_1 file: {proc1_result['output_path']}")
    print(f"proc_1 pre-trim plot: {proc1_result['pre_trim_plot']}")
    print(f"proc_1 post-trim plot: {proc1_result['post_trim_plot']}")
    if show_plots:
        if Image is None:
            print('Inline plot display is unavailable in this environment; open the saved PNG files directly.')
        else:
            display(Image(filename=proc1_result['pre_trim_plot']))
            display(Image(filename=proc1_result['post_trim_plot']))
else:
    print('proc_1 skipped.')


In [ ]:
if run_proc1_step:
    workflow_config = require_ready_config()
    # Expected outputs: a trimmed proc_1 NetCDF plus pre/post deployment review plots.
    proc1_result = run_proc1(workflow_config)
    summary['proc1'] = proc1_result['output_path']
    print(f"proc_1 file: {proc1_result['output_path']}")
    print(f"proc_1 pre-trim plot: {proc1_result['pre_trim_plot']}")
    print(f"proc_1 post-trim plot: {proc1_result['post_trim_plot']}")
    if show_plots:
        display(Image(filename=proc1_result['pre_trim_plot']))
        display(Image(filename=proc1_result['post_trim_plot']))
else:
    print('proc_1 skipped.')


## Run `proc_2`

This step applies `manual_qc_flags` to the already-trimmed `proc_1` output and writes the QC log used to document operator edits.

> `proc_2` uses already-trimmed `proc_1` data and does not trim again.


In [ ]:
if run_proc2_step:
    workflow_config = require_ready_config()
    proc2_input = proc1_result['output_path'] if proc1_result else None
    proc2_result = run_proc2(workflow_config, input_dataset=proc2_input)
    summary['proc2'] = proc2_result['output_path']
    summary['qc_log'] = proc2_result['manual_qc_log']
    print(f"proc_2 file: {proc2_result['output_path']}")
    print(f"QC log: {proc2_result['manual_qc_log']}")
else:
    print('proc_2 skipped.')


## Run `imos_delivery`

This step publishes FV00 from `proc_1` and FV01 from `proc_2`, using the files already recorded in metadata when earlier steps are skipped.


In [ ]:
if run_delivery_step:
    workflow_config = require_ready_config()
    delivery_result = run_imos_delivery(workflow_config)
    summary['fv00'] = delivery_result['proc_1_delivery']
    summary['fv01'] = delivery_result['proc_2_delivery']
    print(f"FV00 path: {delivery_result['proc_1_delivery']}")
    print(f"FV01 path: {delivery_result['proc_2_delivery']}")
else:
    print('IMOS delivery skipped.')


## Troubleshooting

- **Missing `inst_deploy_id`**: use the metadata preview cell to find the AQD deployment identifier, then rerun the config and workflow cells.
- **Missing `proc_1` file when running `proc_2`**: run `proc_1` first, or confirm the metadata row already points to a valid `proc_1_file` in `proc_1_path`.
- **Empty metadata preview filters**: clear `show_year` and `show_location`, or confirm the metadata CSV contains AQD rows for those filters.


## Execution summary

This final cell prints the key output paths collected from the steps you ran in this session.


In [ ]:
print(summary)
